# s1-ready: automatic-labeling end-to-end experiment

This notebook tests the library's **headline promise**: that `s1-ready` can
label an *unlabeled* corpus well enough to fine-tune a usable decision model.

It is deliberately stricter than the main SMS case study (which trained on the
corpus's human labels). Here we **hide** the human train labels, let `s1-ready`
**generate** them with LLM teachers, and only use the human labels as an answer
key. Four models are compared on the **same untouched human-labeled test set**:

| arm | train labels |
|---|---|
| Base | none (zero-shot) |
| **FT-auto** | **s1-ready generated** |
| FT-human | human gold (reference) |
| Classical | human gold (TF-IDF + NB) |

**The number that matters: FT-human F1 − FT-auto F1 — the *price of automation*.**
Near zero means s1-ready's labels are as good as human labels for training.

Requires: a GPU runtime (Runtime → Change runtime type → T4 GPU) **and** an LLM
API key for the labeling step.

## 1. Install

In [ ]:
!pip install -q unsloth trl datasets scikit-learn
# s1-ready itself (zero-dep core + the LLM teacher extra for labeling):
!pip install -q "s1-ready[llm] @ git+https://github.com/mpetrenk/s1-ready.git"

## 2. Get the corpus + API key

Download the UCI SMS Spam Collection, and set the teacher key. The key is read
from the environment — paste it into the Colab **Secrets** panel (key icon) as
`OPENAI_API_KEY` and enable it for this notebook, or set it below. Never commit a key.

In [ ]:
import os
# From Colab Secrets (preferred):
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass  # or set os.environ['OPENAI_API_KEY'] = '...' manually

!wget -q https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip -O sms.zip
!unzip -o -q sms.zip
!ls -la SMSSpamCollection

## 3. Auto-label with s1-ready, and measure label quality

This is the step under test. `label_sms.py` hides the human train labels, has
s1-ready generate labels from raw text, scores them against the hidden gold, and
writes three datasets: `train_auto.jsonl` (s1 labels), `train_human.jsonl`
(reference), `test_human.jsonl` (shared, untouched).

Start with `--limit 300` to bound API cost; drop it for the full run.

In [ ]:
!curl -sL https://raw.githubusercontent.com/mpetrenk/s1-ready/main/examples/case_study/auto_label/label_sms.py -o label_sms.py
!curl -sL https://raw.githubusercontent.com/mpetrenk/s1-ready/main/examples/case_study/auto_label/eval_four_way.py -o eval_four_way.py

# Two-teacher consensus if you also set ANTHROPIC_API_KEY; one teacher is fine.
!python label_sms.py SMSSpamCollection --teacher openai:gpt-4o-mini --out-dir out --limit 300

import json
print(json.dumps(json.load(open('out/label_quality.json')), indent=2))

**Read `label_quality.json` before training.** `f1_spam` under
`label_quality_all_rows` is how well s1-ready's generated labels match the hidden
human labels. `label_quality_after_review` shows the same after dropping
low-agreement rows — i.e. how much a quick human review would improve them. If
label quality is poor here, the downstream fine-tune cannot be good; that is the
honest early signal.

## 4. Fine-tune one arm (reusable function)

LoRA, 1 epoch, 4-bit Llama-3.1-8B — a few minutes on a T4. We call this twice:
once on `train_auto.jsonl`, once on `train_human.jsonl`. Identical recipe, so the
*only* difference between the two models is label source.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import standardize_sharegpt, get_chat_template
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "unsloth/llama-3.1-8b-instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

def load_base():
    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)
    tok = get_chat_template(tok, chat_template='llama-3.1')
    return model, tok

def finetune(train_path, out_dir):
    model, tok = load_base()
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    ds = load_dataset('json', data_files=train_path, split='train')
    ds = standardize_sharegpt(ds)
    def fmt(ex):
        return {'text': tok.apply_chat_template(ex['conversations'], tokenize=False, add_generation_prompt=False)}
    ds = ds.map(fmt)
    trainer = SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds,
        args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                       warmup_steps=5, num_train_epochs=1, learning_rate=2e-4,
                       logging_steps=10, optim='adamw_8bit', seed=7,
                       output_dir=out_dir, dataset_text_field='text'))
    trainer.train()
    return model, tok

## 5. Prediction helper (used by every LLM arm)

One function turns a loaded model into `predict_fn(texts) -> list[bool]`, exactly
what `eval_four_way.eval_llm` expects. Same prompt for base and both fine-tunes,
so the comparison is fair.

In [ ]:
import re, torch
PROMPT = ('State: {text}\nQuestion: Is this SMS message spam?\n'
          "Answer with only 'true' or 'false'.")

def make_predict(model, tok):
    FastLanguageModel.for_inference(model)
    def predict(texts):
        out = []
        for t in texts:
            msgs = [{'role':'user','content': PROMPT.format(text=t)}]
            ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
            gen = model.generate(input_ids=ids, max_new_tokens=5, do_sample=False)
            ans = tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip().lower()
            out.append(ans.startswith(('t','y','1','spam')))
        return out
    return predict

## 6. Run all four arms on the shared test set

In [ ]:
import eval_four_way as E
TEST = 'out/test_human.jsonl'
results = {}

# Classical (CPU, seconds) — trained on human labels.
results['classical'] = E.eval_classical('out/train_human.jsonl', TEST)
print('classical:', results['classical'])

# Base (zero-shot) — load once, score, then free before fine-tuning.
m, t = load_base(); FastLanguageModel.for_inference(m)
results['base'] = E.eval_llm(make_predict(m, t), TEST)
print('base:', results['base'])
del m, t; torch.cuda.empty_cache()

# FT-auto — fine-tuned on s1-ready's GENERATED labels (the arm under test).
m, t = finetune('out/train_auto.jsonl', 'ft_auto')
results['ft_auto'] = E.eval_llm(make_predict(m, t), TEST)
print('ft_auto:', results['ft_auto'])
del m, t; torch.cuda.empty_cache()

# FT-human — same recipe, human labels (reference).
m, t = finetune('out/train_human.jsonl', 'ft_human')
results['ft_human'] = E.eval_llm(make_predict(m, t), TEST)
print('ft_human:', results['ft_human'])
del m, t; torch.cuda.empty_cache()

## 7. Verdict

In [ ]:
import json
print(E.summarize(results))
json.dump(results, open('out/four_way_results.json','w'), indent=2)
print('\nsaved out/four_way_results.json — commit this + label_quality.json back to the repo')

**Interpreting the verdict.**
- `ft_auto` vs `ft_human`: the **price of automation**. Small gap → s1-ready's
  labels are good enough to train on (the thesis holds).
- `ft_auto` vs `base`: did fine-tuning on auto-labels help at all?
- `ft_auto` vs `classical`: is the whole LLM route worth it over a 20-line baseline
  on this (easy) task? Expect it to be close here — the value shows on harder tasks.
- Cross-check against `label_quality.json`: downstream F1 should track label quality.